## Widgets

In [0]:
%python
dbutils.widgets.removeAll()

#remove bidgets

In [0]:
%sql
--- crear widgets

create widget text storageName default "adlssmartprojectdev13";
create widget text catalogo default "project_dev";
create widget text container default "raw";
create widget text schemaBronze default "bronze";
create widget text schemaSilver default "silver";
create widget text schemaGold default "gold";
create widget text credential_ default "credential_dev";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")
container = dbutils.widgets.get("container")
schemaBronze = dbutils.widgets.get("schemaBronze")
schemaSilver = dbutils.widgets.get("schemaSilver")
schemaGold = dbutils.widgets.get("schemaGold")
catalogo = dbutils.widgets.get("catalogo")
credential_ = dbutils.widgets.get("credential_")


# guardar el widget en una variable

##Catalogs y Schemas

In [0]:
%sql
--- crear el catalogo

CREATE CATALOG IF NOT EXISTS ${catalogo};

In [0]:
%sql
---  crear los eschemas y volume necesario

CREATE SCHEMA IF NOT EXISTS ${catalogo}.${container};
CREATE SCHEMA IF NOT EXISTS ${catalogo}.${schemaBronze};
CREATE SCHEMA IF NOT EXISTS ${catalogo}.${schemaSilver};
CREATE SCHEMA IF NOT EXISTS ${catalogo}.${schemaGold};

CREATE VOLUME IF NOT EXISTS ${catalogo}.${container}.datasets;

## external locations

In [0]:
%sql
---- eliminar external locations

/*DROP EXTERNAL LOCATION IF EXISTS `exlt-raw-data`;
DROP EXTERNAL LOCATION IF EXISTS `exlt-bronze` ;
DROP EXTERNAL LOCATION IF EXISTS `exlt-silver`;
DROP EXTERNAL LOCATION IF EXISTS `exlt-gold`;*/

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw-data`
URL 'abfss://${container}@${storageName}.dfs.core.windows.net/raw_data_project/'
WITH (STORAGE CREDENTIAL ${credential_})
COMMENT 'Ubicación externa para las tablas raw del Data Lake en la carpeta raw_data_project';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://${schemaBronze}@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL ${credential_})
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://${schemaSilver}@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL ${credential_})
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-gold`
URL 'abfss://gold@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL ${credential_})
COMMENT 'Ubicación externa para las tablas gold del Data Lake';

#TABLAS BRONZE

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.${schemaBronze}.orders_items (
order_id String,
order_item_id Integer,
product_id String,
seller_id String,
shipping_limit_date Timestamp,
price Decimal(10,2),
freight_value Decimal(10,2)
)
USING DELTA
--PARTITIONED BY (order_id)
LOCATION "abfss://${schemaBronze}@${storageName}.dfs.core.windows.net/orders_items"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.${schemaBronze}.orders_payments (
order_id String,
payment_sequential Integer,
payment_type String,
payment_installments String,
payment_value Decimal(10,2)
)
USING DELTA

LOCATION "abfss://${schemaBronze}@${storageName}.dfs.core.windows.net/orders_payments"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.${schemaBronze}.orders_review (
order_id String,
review_score String,
review_creation_date String,
review_answer_timestamp String
)
USING DELTA

LOCATION "abfss://${schemaBronze}@${storageName}.dfs.core.windows.net/orders_review"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.${schemaBronze}.orders (
order_id String,
customer_id String,
order_status String,
order_purchase_timestamp Timestamp,
order_approved_at Timestamp,
order_delivered_carrier_date Timestamp,
order_delivered_customer_date Timestamp,
order_estimated_delivery_date Timestamp
)
USING DELTA

LOCATION "abfss://${schemaBronze}@${storageName}.dfs.core.windows.net/orders"


In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.${schemaBronze}.products (
product_id string ,
product_category_name string,
product_name_length integer,
product_description_length integer,
product_photos_qty integer,
product_weight_g integer,
product_length_cm integer,
product_height_cm integer,
product_width_cm integer
)
USING DELTA

LOCATION "abfss://${schemaBronze}@${storageName}.dfs.core.windows.net/products"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.${schemaBronze}.products_category (
product_category_name string ,
product_category_name_english string
)
USING DELTA

LOCATION "abfss://${schemaBronze}@${storageName}.dfs.core.windows.net/products_category"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.${schemaBronze}.customers (
customer_id String, 
customer_unique_id String,
customer_zip_code_prefix Integer,
customer_city String,
customer_state String
)
USING DELTA

LOCATION "abfss://${schemaBronze}@${storageName}.dfs.core.windows.net/customers"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.${schemaBronze}.sellers (
seller_id string, 
seller_zip_code_prefix string, 
seller_city string, 
seller_state string
)
USING DELTA

LOCATION "abfss://${schemaBronze}@${storageName}.dfs.core.windows.net/sellers"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.${schemaBronze}.geolocation (
geolocation_zip_code_prefix string,
geolocation_lat string,
geolocation_lng string,
geolocation_city string,
geolocation_state string
)
USING DELTA

LOCATION "abfss://${schemaBronze}@${storageName}.dfs.core.windows.net/geolocation"

## tablas silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.${schemaSilver}.orders_transform (
order_id string,
order_item_id int,
product_id string, 
seller_id string, 
shipping_limit_date timestamp, 
boleto_price_end decimal(10,3),
boleto_freight_end decimal(10,3), 
not_defined_price_end decimal(10,3), 
not_defined_freight_end decimal(10,3), 
credit_card_price_end decimal(10,3), 
credit_card_freight_end decimal(10,3), 
voucher_price_end decimal(10,3), 
voucher_freight_end decimal(10,3), 
debit_card_price_end decimal(10,3), 
debit_card_freight_end decimal(10,3), 
customer_id string, 
order_status string, 
order_approved_at timestamp, 
order_delivered_carrier_date timestamp
)
USING DELTA

LOCATION "abfss://${schemaSilver}@${storageName}.dfs.core.windows.net/orders_transform"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.${schemaSilver}.products_transform (
product_id string,
product_category_name string, 
product_category_name_english string 
)
USING DELTA

LOCATION "abfss://${schemaSilver}@${storageName}.dfs.core.windows.net/products_transform"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.${schemaSilver}.customers (
customer_id STRING, 
customer_city STRING,
customer_state STRING
)
USING DELTA

LOCATION "abfss://${schemaSilver}@${storageName}.dfs.core.windows.net/customers"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.${schemaSilver}.sellers (
seller_id STRING, 
seller_city STRING,
seller_state STRING
)
USING DELTA

LOCATION "abfss://${schemaSilver}@${storageName}.dfs.core.windows.net/sellers"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.${schemaSilver}.orders_transform_end (
order_id string,
order_item_id int,
product_id string, 
seller_id string, 
shipping_limit_date timestamp,  
customer_id string, 
order_status string, 
order_approved_at timestamp, 
order_delivered_carrier_date timestamp,
product_category_name string, 
product_category_name_english string, 
seller_city string, 
seller_state string, 
customer_city string, 
customer_state string,
payment_type string,
price decimal(10,3),
freight decimal(10,3)
)
USING DELTA

LOCATION "abfss://${schemaSilver}@${storageName}.dfs.core.windows.net/orders_transform_end"

## tabla gold

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.${schemaGold}.orders (
order_id string, 
order_item_id int, 
product_id string, 
order_status string, 
order_delivered_carrier_date date, 
product_category_name string, 
product_category_name_english string, 
seller_city string, 
seller_state string, 
customer_city string, 
customer_state string, 
payment_type string, 
price decimal(10,3), 
freight decimal(10,3),
customer_id string,
seller_id string
)
USING DELTA

LOCATION "abfss://${schemaGold}@${storageName}.dfs.core.windows.net/orders"